In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None) 
import numpy as np
#pd.set_option('display.max_info_columns', None)


In [2]:
# Function to get the name of a DataFrame
def get_variable_name(df):
    for name, value in globals().items():
        if value is df:
            return name


def count_repetitive_caseids(df):
    if 'CASEID' in df.columns:  # Check if 'CASEID' column exists
        duplicate_count = df['CASEID'].duplicated().sum()  # Count duplicates
        total_count = df['CASEID'].count()  # Count total rows
        unique_count = df['CASEID'].nunique()  # Count unique CASEID
        return duplicate_count, total_count, unique_count
    else:
        return None, None, None  # If 'CASEID' column is missing
    


def add_prefix_except_caseid(df, prefix):
    """
    Add a prefix to all columns in the dataframe except 'CASEID'.

    Parameters:
        df (pd.DataFrame): The input dataframe.
        prefix (str): The prefix to add.

    Returns:
        pd.DataFrame: A new dataframe with prefixed column names.
    """
    df = df.copy()
    new_columns = {
        col: f"{prefix}_{col}" if col != 'CASEID' else col
        for col in df.columns
    }
    return df.rename(columns=new_columns)

def get_dummy_variables(df):
    """
    Identify dummy (binary) variables in a DataFrame.
    A dummy variable is defined as having only 0, 1, or NaN values.
    Columns that are entirely NaN are excluded.

    Parameters:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        List[str]: List of column names that are dummy variables.
    """
    dummy_cols = []
    for col in df.columns:
        # Skip if all values are NaN
        if df[col].isna().all():
            continue
        
        unique_vals = df[col].dropna().unique()
        if set(unique_vals).issubset({0, 1}):
            dummy_cols.append(col)
    return dummy_cols



def convert_objects_to_int64_safe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert object columns in a DataFrame to Int64 if possible.
    If conversion is not possible (non-integer strings), keep the column as object.
    """
    df_converted = df.copy()

    for col in df_converted.select_dtypes(include=["object"]).columns:
        s = df_converted[col].astype(str).str.strip()   # remove extra spaces
        s = s.mask(s == "", np.nan)                     # empty string -> NaN (no downcasting warning)
        
        # Try converting
        try:
            converted = pd.to_numeric(s, errors="raise").astype("Int64")
            df_converted[col] = converted
        except Exception:
            df_converted[col] = df_converted[col]  # keep original if fails
    
    return df_converted

def drop_null_and_list(df, exclude_list):
    """
    Drop columns that are entirely NaN or present in the exclude_list.

    Parameters:
        df (pd.DataFrame): Input DataFrame.
        exclude_list (list): List of column names to exclude.

    Returns:
        pd.DataFrame: DataFrame with specified columns removed.
    """
    # Drop all-null columns
    df_filtered = df.dropna(axis=1, how="all")
    
    # Drop columns from exclude_list
    df_filtered = df_filtered.drop(columns=[col for col in exclude_list if col in df_filtered.columns], errors="ignore")
    
    return df_filtered

def aggregate_sum_by_caseid(df, caseid_col="CASEID"):
    """
    Aggregate a DataFrame at the CASEID level by summing numeric columns.

    Parameters:
        df (pd.DataFrame): Input DataFrame.
        caseid_col (str): Name of the column that identifies the case/group.

    Returns:
        pd.DataFrame: Aggregated DataFrame with one row per CASEID.
    """
    # Group by CASEID and sum only numeric columns
    aggregated_df = (
        df.groupby(caseid_col, as_index=False)
          .sum(numeric_only=True)
    )
    return aggregated_df


def aggregate_with_value_suffix(df, caseid_col="CASEID", exclude_cols=None):
    """
    Aggregate at CASEID level by creating dummy columns for each unique value in selected columns,
    and summing the counts. Excludes columns specified in exclude_cols.

    Parameters:
        df (pd.DataFrame): Input DataFrame
        caseid_col (str): Column name to group by
        exclude_cols (list): List of columns to exclude from processing

    Returns:
        pd.DataFrame: Aggregated DataFrame with CASEID and dummy count columns
    """
    if exclude_cols is None:
        exclude_cols = []

    # Columns to process: all except caseid_col and excluded ones
    cols_to_process = [c for c in df.columns if c not in [caseid_col] + exclude_cols]

    # Create dummy variables for the selected columns
    df_dummies = pd.get_dummies(df[cols_to_process].astype(str), prefix=cols_to_process)

    # Combine CASEID with the dummies
    df_combined = pd.concat([df[[caseid_col]], df_dummies], axis=1)

    # Aggregate by CASEID summing the dummy counts
    agg_df = df_combined.groupby(caseid_col, as_index=False).sum()

    return agg_df


def aggregate_by_caseid_mean(df, caseid_col="CASEID"):
    """
    Aggregate a DataFrame by caseid_col and compute the mean of all other columns.
    
    Parameters:
    - df (pd.DataFrame): Input DataFrame
    - caseid_col (str): Column to group by (default = "CASEID")
    
    Returns:
    - pd.DataFrame: Aggregated DataFrame with mean values
    """
    # Group by CASEID and calculate mean for numeric columns
    df_agg = df.groupby(caseid_col, as_index=False).mean(numeric_only=True)
    
    return df_agg


def analyze_matches(left_df, right_df, label):
    original_rows = len(left_df)
    right_rows = len(right_df)
    
    merged = left_df.merge(right_df, on='CASEID', how='left', indicator=True)
    merged_rows = len(merged)

    match_count = (merged['_merge'] == 'both').sum()
    match_pct = match_count / original_rows * 100

    print(f"🔎 {label}")
    print(f" - Rows in target_final: {original_rows}")
    print(f" - Rows in right dataframe ({label}): {right_rows}")
    print(f" - Rows after merge: {merged_rows}")
    print(f" ✅ Matches on CASEID: {match_count} ({match_pct:.2f}%)\n")


def analyze_matches_explicit_keys(target_df, right_df, label, right_key):
    # Drop duplicates in right dataframe based on CASEID and the specific key
    right_df_clean = right_df.drop_duplicates(subset=['CASEID', right_key])

    # Perform merge using explicit keys
    merged = target_df.merge(
        right_df_clean,
        left_on=['CASEID', 'BIDX'],
        right_on=['CASEID', right_key],
        how='left',
        indicator=True
    )

    # Count and summarize
    original_rows = len(target_df)
    right_rows = len(right_df)
    merged_rows = len(merged)
    match_count = (merged['_merge'] == 'both').sum()
    match_pct = match_count / original_rows * 100

    # Print results
    print(f"🔎 {label}")
    print(f" - Rows in target_final: {original_rows}")
    print(f" - Rows in right dataframe: {right_rows}")
    print(f" - Matches on CASEID + {right_key}: {match_count} ({match_pct:.2f}%)\n")

# 01 Target

In [3]:
Modulo1632_REC21_2023 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1632/968-Modulo1632/REC21_2024.csv", low_memory=False)
Modulo1632_REC21_2023.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60693 entries, 0 to 60692
Data columns (total 34 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   ID1      60693 non-null  int64 
 1   CASEID   60693 non-null  object
 2   BIDX     60693 non-null  int64 
 3   BORD     60693 non-null  int64 
 4   B0       60693 non-null  int64 
 5   B1       60693 non-null  int64 
 6   B2       60693 non-null  int64 
 7   B3       60693 non-null  object
 8   B4       60693 non-null  int64 
 9   B5       60693 non-null  int64 
 10  B6       60693 non-null  object
 11  B7       60693 non-null  object
 12  B8       60693 non-null  object
 13  B9       60693 non-null  object
 14  B10      60693 non-null  int64 
 15  B11      60693 non-null  object
 16  B12      60693 non-null  object
 17  B13      60693 non-null  object
 18  B15      60693 non-null  object
 19  B16      60693 non-null  object
 20  BD       60693 non-null  int64 
 21  BDD      60693 non-null  int64 
 22

In [4]:
target = Modulo1632_REC21_2023[['CASEID', 'BIDX','Q220A']]
target.drop_duplicates().shape

(60693, 3)

In [5]:
# # de meses de nascimento
target.Q220A.value_counts()

Q220A
     39575
9    16932
8     3467
7      580
6      122
5       17
Name: count, dtype: int64

In [6]:
target=target[target['Q220A']!= " " ]
target.Q220A.value_counts()

Q220A
9    16932
8     3467
7      580
6      122
5       17
Name: count, dtype: int64

In [7]:
target.BIDX.value_counts()

BIDX
1    18335
2     2637
3      143
4        3
Name: count, dtype: int64

In [8]:

target.shape, target.CASEID.nunique()

# come:numero de CASEID y numero de CASEID unicos. O sea, un CASEID puede tener mais de un hijo en la encuenta.   

((21118, 3), 18335)

In [16]:
target['Q220A'] = pd.to_numeric(target['Q220A'],)

target.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21118 entries, 0 to 60690
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   CASEID  21118 non-null  object
 1   BIDX    21118 non-null  int64 
 2   Q220A   21118 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 659.9+ KB


In [17]:
target.shape,  target.CASEID.nunique()

((21118, 3), 18335)

In [18]:
target['premature'] =  np.where(target['Q220A'] <= 8, 1, 0)
target_final = target.groupby('CASEID', as_index=False)['premature'].sum()
target_final['premature_flag'] = np.where(target_final['premature'] >= 1, 1, 0)
target_final.premature_flag.value_counts()


premature_flag
0    14517
1     3818
Name: count, dtype: int64

In [19]:
3818/(3818+ 14517)

0.20823561494409598

In [20]:
target_final.info(verbose = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   CASEID          18335 non-null  object
 1   premature       18335 non-null  int32 
 2   premature_flag  18335 non-null  int32 
dtypes: int32(2), object(1)
memory usage: 286.6+ KB


In [21]:
target_final.premature_flag.value_counts(1)
# come: 20% tuvieron al menos 1 parto prematuro. 

premature_flag
0    0.791764
1    0.208236
Name: proportion, dtype: float64

In [60]:
# File name only
output_file = "target_final.csv"

# Go one level up from the current working directory
base_dir = os.path.dirname(os.getcwd())   # gives "c:\\Users\\linoc\\OneDrive\\Encoder\\03_partos"
output_dir = os.path.join(base_dir, "03_bases_intermediarios")

#Full path
output_path = os.path.join(output_dir, output_file)

# Save DataFrame
target_final.to_csv(output_path, index=False, encoding="utf-8-sig")

# 2 join other tables

In [26]:
Modulo1631_REC91_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1631/968-Modulo1631/REC91_2024.csv", low_memory=False)
Modulo1632_RE223132_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1632/968-Modulo1632/RE223132_2024.csv", low_memory=False)
Modulo1633_REC41_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1633/968-Modulo1633/REC41_2024.csv", low_memory=False)
Modulo1633_REC94_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1633/968-Modulo1633/REC94_2024.csv", low_memory=False)
Modulo1635_RE516171_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1635/968-Modulo1635/RE516171_2024.csv", low_memory=False)
Modulo1640_CSALUD01_2024 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1640/968-Modulo1640/CSALUD01_2024.csv", low_memory=False)

print(f"size file: {get_variable_name(Modulo1631_REC91_2024)}: {Modulo1631_REC91_2024.shape}")
print(f"size file: {get_variable_name(Modulo1632_RE223132_2024)}: {Modulo1632_RE223132_2024.shape}")
print(f"size file: {get_variable_name(Modulo1633_REC41_2024)}: {Modulo1633_REC41_2024.shape}")
print(f"size file: {get_variable_name(Modulo1633_REC94_2024)}: {Modulo1633_REC94_2024.shape}")
print(f"size file: {get_variable_name(Modulo1635_RE516171_2024)}: {Modulo1635_RE516171_2024.shape}")
print(f"size file: {get_variable_name(Modulo1640_CSALUD01_2024)}: {Modulo1640_CSALUD01_2024.shape}")


size file: Modulo1631_REC91_2024: (37117, 343)
size file: Modulo1632_RE223132_2024: (34252, 149)
size file: Modulo1633_REC41_2024: (19751, 147)
size file: Modulo1633_REC94_2024: (19751, 61)
size file: Modulo1635_RE516171_2024: (34252, 84)
size file: Modulo1640_CSALUD01_2024: (34018, 258)


## 2.1 match with CASEID level 

In [27]:
Modulo1631_REC91_2024 = Modulo1631_REC91_2024.drop_duplicates()
Modulo1632_RE223132_2024 = Modulo1632_RE223132_2024.drop_duplicates()
Modulo1635_RE516171_2024 = Modulo1635_RE516171_2024.drop_duplicates()

In [28]:
Modulo1631_REC91_2024.shape,Modulo1632_RE223132_2024.shape, Modulo1635_RE516171_2024.shape

((37117, 343), (34252, 149), (34252, 84))

In [29]:
analyze_matches(target_final, Modulo1631_REC91_2024, 'Modulo1631_REC91_2024')
analyze_matches(target_final, Modulo1632_RE223132_2024, 'Modulo1632_RE223132_2024')
analyze_matches(target_final, Modulo1635_RE516171_2024, 'Modulo1635_RE516171_2024')


🔎 Modulo1631_REC91_2024
 - Rows in target_final: 18335
 - Rows in right dataframe (Modulo1631_REC91_2024): 37117
 - Rows after merge: 18335
 ✅ Matches on CASEID: 18335 (100.00%)

🔎 Modulo1632_RE223132_2024
 - Rows in target_final: 18335
 - Rows in right dataframe (Modulo1632_RE223132_2024): 34252
 - Rows after merge: 18335
 ✅ Matches on CASEID: 18335 (100.00%)

🔎 Modulo1635_RE516171_2024
 - Rows in target_final: 18335
 - Rows in right dataframe (Modulo1635_RE516171_2024): 34252
 - Rows after merge: 18335
 ✅ Matches on CASEID: 18335 (100.00%)



In [30]:
print(Modulo1631_REC91_2024.shape[0]), print(Modulo1631_REC91_2024.CASEID.nunique())
print(Modulo1632_RE223132_2024.shape[0]), print(Modulo1632_RE223132_2024.CASEID.nunique())
print(Modulo1635_RE516171_2024.shape[0]), print(Modulo1635_RE516171_2024.CASEID.nunique())

37117
37117
34252
34252
34252
34252


(None, None)

In [31]:
Modulo1631_REC91_2024 = add_prefix_except_caseid(Modulo1631_REC91_2024, 'REC91')
Modulo1632_RE223132_2024 = add_prefix_except_caseid(Modulo1631_REC91_2024, 'RE223132')
Modulo1635_RE516171_2024 = add_prefix_except_caseid(Modulo1635_RE516171_2024, 'RE516171')

# Perform left merges one by one
target_merged = target_final.merge(Modulo1631_REC91_2024, on='CASEID', how='left') \
                            .merge(Modulo1632_RE223132_2024, on='CASEID', how='left') \
                            .merge(Modulo1635_RE516171_2024, on='CASEID', how='left')
target_merged.info(verbose  = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18335 entries, 0 to 18334
Data columns (total 770 columns):
 #    Column                   Non-Null Count  Dtype 
---   ------                   --------------  ----- 
 0    CASEID                   18335 non-null  object
 1    premature                18335 non-null  int32 
 2    premature_flag           18335 non-null  int32 
 3    REC91_ID1                18335 non-null  int64 
 4    REC91_SVER               18335 non-null  int64 
 5    REC91_SREGION            18335 non-null  int64 
 6    REC91_SSEMES             18335 non-null  int64 
 7    REC91_SPROVIN            18335 non-null  int64 
 8    REC91_SDISTRI            18335 non-null  int64 
 9    REC91_S108N              18335 non-null  object
 10   REC91_S108Y              18335 non-null  object
 11   REC91_S108G              18335 non-null  object
 12   REC91_S111               18335 non-null  object
 13   REC91_S112               18335 non-null  object
 14   REC91_S119          

## 2.2 match with CASEID and BIRD

In [37]:
target = target.drop_duplicates()
target.shape

(21118, 4)

In [38]:
Modulo1633_REC41_2024.shape, Modulo1633_REC94_2024.shape

((19751, 147), (19751, 61))

In [39]:
#Modulo1632_REC21_2024 = Modulo1632_REC21_2024.drop_duplicates()
Modulo1633_REC41_2024 = Modulo1633_REC41_2024.drop_duplicates()
Modulo1633_REC94_2024  = Modulo1633_REC94_2024.drop_duplicates()


In [40]:
Modulo1633_REC41_2024.shape, Modulo1633_REC94_2024.shape

((19751, 147), (19751, 61))

In [41]:
#analyze_matches_explicit_keys(target, Modulo1632_REC21_2024, 'Modulo1632_REC21_2024', 'BIDX')
analyze_matches_explicit_keys(target, Modulo1633_REC41_2024, 'Modulo1633_REC41_2024', 'MIDX')
analyze_matches_explicit_keys(target, Modulo1633_REC94_2024, 'Modulo1633_REC94_2024', 'IDX94')


🔎 Modulo1633_REC41_2024
 - Rows in target_final: 21118
 - Rows in right dataframe: 19751
 - Matches on CASEID + MIDX: 19751 (93.53%)

🔎 Modulo1633_REC94_2024
 - Rows in target_final: 21118
 - Rows in right dataframe: 19751
 - Matches on CASEID + IDX94: 19751 (93.53%)



### 2.2.1 Modulo1633_REC41_2024

In [42]:
Modulo1633_REC41_2024_M = convert_objects_to_int64_safe(Modulo1633_REC41_2024)
Modulo1633_REC41_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19751 entries, 0 to 19750
Data columns (total 147 columns):
 #    Column  Non-Null Count  Dtype 
---   ------  --------------  ----- 
 0    ID1     19751 non-null  int64 
 1    CASEID  19751 non-null  object
 2    MIDX    19751 non-null  int64 
 3    M1      17608 non-null  Int64 
 4    M1A     8385 non-null   Int64 
 5    M1B     6079 non-null   Int64 
 6    M1C     6079 non-null   Int64 
 7    M1D     534 non-null    Int64 
 8    M1E     6079 non-null   Int64 
 9    M2A     17608 non-null  Int64 
 10   M2B     17608 non-null  Int64 
 11   M2C     17608 non-null  Int64 
 12   M2D     17608 non-null  Int64 
 13   M2E     17608 non-null  Int64 
 14   M2F     0 non-null      Int64 
 15   M2G     17608 non-null  Int64 
 16   M2H     0 non-null      Int64 
 17   M2I     0 non-null      Int64 
 18   M2J     0 non-null      Int64 
 19   M2K     17608 non-null  Int64 
 20   M2L     0 non-null      Int64 
 21   M2M     0 non-null      Int64 
 2

In [44]:
dummy_variables  = get_dummy_variables(Modulo1633_REC41_2024_M)
Modulo1633_REC41_2024_M[dummy_variables].describe().T

,count,mean,std,min,25%,50%,75%,max
M2A,17608.0,0.252385,0.434393,0.0,0.0,0.0,1.0,1.0
M2B,17608.0,0.075193,0.26371,0.0,0.0,0.0,0.0,1.0
M2C,17608.0,0.888346,0.314949,0.0,1.0,1.0,1.0,1.0
M2D,17608.0,0.020275,0.140943,0.0,0.0,0.0,0.0,1.0
M2E,17608.0,0.000114,0.010657,0.0,0.0,0.0,0.0,1.0
M2G,17608.0,0.00017,0.013052,0.0,0.0,0.0,0.0,1.0
M2K,17608.0,0.000398,0.019935,0.0,0.0,0.0,0.0,1.0
M2N,17608.0,0.010336,0.101143,0.0,0.0,0.0,0.0,1.0
M3A,19751.0,0.663612,0.472485,0.0,0.0,1.0,1.0,1.0
M3B,19751.0,0.774087,0.418192,0.0,1.0,1.0,1.0,1.0


In [45]:
no_dummy = drop_null_and_list(Modulo1633_REC41_2024_M, dummy_variables)
no_dummy_list =no_dummy.columns.to_list()
no_dummy_list

remove_items = {"M1C", "M1E", "M4","M46","M34", "M19","M6","M7","M8","M9","M11","M13","M14"}
no_dummy_list = [col for col in no_dummy_list if col not in remove_items]


In [46]:
df_dummy_variables = aggregate_sum_by_caseid(Modulo1633_REC41_2024_M[dummy_variables + ['CASEID']])
df_dummy_variables.describe().T

,count,mean,std,min,25%,50%,75%,max
M2A,17608.0,0.252385,0.434393,0.0,0.0,0.0,1.0,1.0
M2B,17608.0,0.075193,0.26371,0.0,0.0,0.0,0.0,1.0
M2C,17608.0,0.888346,0.314949,0.0,1.0,1.0,1.0,1.0
M2D,17608.0,0.020275,0.140943,0.0,0.0,0.0,0.0,1.0
M2E,17608.0,0.000114,0.010657,0.0,0.0,0.0,0.0,1.0
M2G,17608.0,0.00017,0.013052,0.0,0.0,0.0,0.0,1.0
M2K,17608.0,0.000398,0.019935,0.0,0.0,0.0,0.0,1.0
M2N,17608.0,0.010336,0.101143,0.0,0.0,0.0,0.0,1.0
M3A,17608.0,0.744378,0.559132,0.0,0.0,1.0,1.0,4.0
M3B,17608.0,0.868299,0.515318,0.0,1.0,1.0,1.0,4.0


In [47]:
df_no_dummy_list = aggregate_with_value_suffix(Modulo1633_REC41_2024_M[no_dummy_list], caseid_col="CASEID")
df_no_dummy_list.describe().T

,count,mean,std,min,25%,50%,75%,max
ID1_2024,17608.0,1.121706,0.342896,1.0,1.0,1.0,1.0,4.0
MIDX_1,17608.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
MIDX_2,17608.0,0.116538,0.320878,0.0,0.0,0.0,0.0,1.0
MIDX_3,17608.0,0.004998,0.070520,0.0,0.0,0.0,0.0,1.0
MIDX_4,17608.0,0.000170,0.013052,0.0,0.0,0.0,0.0,1.0
M1_0,17608.0,0.184234,0.387686,0.0,0.0,0.0,0.0,1.0
M1_1,17608.0,0.269309,0.443614,0.0,0.0,0.0,1.0,1.0
M1_2,17608.0,0.516811,0.499732,0.0,0.0,1.0,1.0,1.0
M1_3,17608.0,0.006815,0.082274,0.0,0.0,0.0,0.0,1.0
M1_4,17608.0,0.000057,0.007536,0.0,0.0,0.0,0.0,1.0


In [48]:
continue_var_REC41 = ["M6","M7","M8","M9","M11","M13","M14"]

df_continue_var_REC41 = aggregate_by_caseid_mean(Modulo1633_REC41_2024_M[continue_var_REC41 + ['CASEID']], caseid_col="CASEID")
df_continue_var_REC41.head()

,CASEID,M6,M7,M8,M9,M11,M13,M14
0,325503101 2,12.0,12.0,8.0,8.0,<NA>,1.0,6.0
1,325504701 2,4.0,4.0,4.0,4.0,<NA>,4.0,8.0
2,325505001 1,3.0,3.0,3.0,3.0,<NA>,1.0,11.0
3,325508901 2,6.0,6.0,4.0,4.0,<NA>,1.0,12.0
4,325509701 2,96.0,18.0,1.0,1.0,<NA>,2.0,10.0


In [49]:
Modulo1633_REC41_2024_fil = df_dummy_variables.merge(df_no_dummy_list, how='left', on='CASEID')
Modulo1633_REC41_2024_fil =Modulo1633_REC41_2024_fil.merge(df_continue_var_REC41, how = 'left', on = 'CASEID')
Modulo1633_REC41_2024_fil.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17608 entries, 0 to 17607
Data columns (total 467 columns):
 #    Column     Non-Null Count  Dtype  
---   ------     --------------  -----  
 0    CASEID     17608 non-null  object 
 1    M2A        17608 non-null  Int64  
 2    M2B        17608 non-null  Int64  
 3    M2C        17608 non-null  Int64  
 4    M2D        17608 non-null  Int64  
 5    M2E        17608 non-null  Int64  
 6    M2G        17608 non-null  Int64  
 7    M2K        17608 non-null  Int64  
 8    M2N        17608 non-null  Int64  
 9    M3A        17608 non-null  int64  
 10   M3B        17608 non-null  int64  
 11   M3C        17608 non-null  int64  
 12   M3D        17608 non-null  int64  
 13   M3E        17608 non-null  int64  
 14   M3G        17608 non-null  int64  
 15   M3H        17608 non-null  int64  
 16   M3K        17608 non-null  int64  
 17   M3N        17608 non-null  int64  
 18   M17        17608 non-null  int64  
 19   M54        17608 non-nu

In [51]:
import os
# output_path = r"C:\Users\linoc\OneDrive\Encoder\03_partos\03_bases_intermediarios\Modulo1633_REC41_2024_fil.csv"
# Modulo1633_REC41_2024_fil.to_csv(output_path, index=False, encoding='utf-8-sig')

# File name only
output_file = "Modulo1633_REC41_2024_fil.csv"

# Go one level up from the current working directory
base_dir = os.path.dirname(os.getcwd())   # gives "c:\\Users\\linoc\\OneDrive\\Encoder\\03_partos"
output_dir = os.path.join(base_dir, "03_bases_intermediarios")

#Full path
output_path = os.path.join(output_dir, output_file)

# Save DataFrame
Modulo1633_REC41_2024_fil.to_csv(output_path, index=False, encoding="utf-8-sig")


### 2.2.1 Modulo1633_REC94_2024

In [52]:

Modulo1633_REC94_2024_M = convert_objects_to_int64_safe(Modulo1633_REC94_2024)
Modulo1633_REC94_2024_M.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19751 entries, 0 to 19750
Data columns (total 61 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ID1       19751 non-null  int64 
 1   CASEID    19751 non-null  object
 2   IDX94     19751 non-null  int64 
 3   S410B     17334 non-null  Int64 
 4   S411B     17426 non-null  Int64 
 5   S411F     17426 non-null  Int64 
 6   S411G     17426 non-null  Int64 
 7   S411H     17426 non-null  Int64 
 8   S411I     17426 non-null  Int64 
 9   S411J     17426 non-null  Int64 
 10  S411K     17426 non-null  Int64 
 11  S411L     17426 non-null  Int64 
 12  S411BA    17054 non-null  Int64 
 13  S411CA    17141 non-null  Int64 
 14  S411DA    15366 non-null  Int64 
 15  S411EA    16355 non-null  Int64 
 16  S413      17608 non-null  Int64 
 17  S422I     16727 non-null  Int64 
 18  S426B     5993 non-null   Int64 
 19  S426E     6957 non-null   Int64 
 20  S426FA    0 non-null      Int64 
 21  S426FB    15

In [53]:
dummy_variables_REC94  = get_dummy_variables(Modulo1633_REC94_2024_M)
Modulo1633_REC94_2024_M[dummy_variables_REC94].describe().T

,count,mean,std,min,25%,50%,75%,max
S413,17608.0,0.809121,0.393005,0.0,1.0,1.0,1.0,1.0
S426E,6957.0,0.424608,0.494319,0.0,0.0,0.0,1.0,1.0
S426GA,17608.0,0.104498,0.305914,0.0,0.0,0.0,0.0,1.0
S426GB,17608.0,0.089334,0.285234,0.0,0.0,0.0,0.0,1.0
S426GC,17608.0,0.018458,0.134603,0.0,0.0,0.0,0.0,1.0
S426GD,17608.0,0.009144,0.095187,0.0,0.0,0.0,0.0,1.0
S426GE,17608.0,0.048898,0.215661,0.0,0.0,0.0,0.0,1.0
S430D,17431.0,0.998738,0.035505,0.0,1.0,1.0,1.0,1.0
S427DA,17608.0,0.045264,0.207887,0.0,0.0,0.0,0.0,1.0
S427DB,17608.0,0.020275,0.140943,0.0,0.0,0.0,0.0,1.0


In [54]:
df_dummy_variables_REC94 = aggregate_sum_by_caseid(Modulo1633_REC94_2024_M[dummy_variables_REC94 + ['CASEID']])
df_dummy_variables_REC94.describe().T

,count,mean,std,min,25%,50%,75%,max
S413,17608.0,0.809121,0.393005,0.0,1.0,1.0,1.0,1.0
S426E,17608.0,0.167765,0.405591,0.0,0.0,0.0,0.0,3.0
S426GA,17608.0,0.104498,0.305914,0.0,0.0,0.0,0.0,1.0
S426GB,17608.0,0.089334,0.285234,0.0,0.0,0.0,0.0,1.0
S426GC,17608.0,0.018458,0.134603,0.0,0.0,0.0,0.0,1.0
S426GD,17608.0,0.009144,0.095187,0.0,0.0,0.0,0.0,1.0
S426GE,17608.0,0.048898,0.215661,0.0,0.0,0.0,0.0,1.0
S430D,17608.0,0.988698,0.4492,0.0,1.0,1.0,1.0,3.0
S427DA,17608.0,0.045264,0.207887,0.0,0.0,0.0,0.0,1.0
S427DB,17608.0,0.020275,0.140943,0.0,0.0,0.0,0.0,1.0


In [55]:
no_dummy_REC94 = drop_null_and_list(Modulo1633_REC94_2024_M, dummy_variables_REC94)
no_dummy_list_REC94 =no_dummy_REC94.columns.to_list()
remove_items = {"S411BA","S411CA","S411DA","S411EA","S422I"}
no_dummy_list = [col for col in no_dummy_list if col not in remove_items]

In [56]:
df_no_dummy_list_REC94 = aggregate_with_value_suffix(Modulo1633_REC94_2024_M[no_dummy_list_REC94], caseid_col="CASEID")
df_no_dummy_list_REC94.describe().T

,count,mean,std,min,25%,50%,75%,max
ID1_2024,17608.0,1.121706,0.342896,1.0,1.0,1.0,1.0,4.0
IDX94_1,17608.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
IDX94_2,17608.0,0.116538,0.320878,0.0,0.0,0.0,0.0,1.0
IDX94_3,17608.0,0.004998,0.070520,0.0,0.0,0.0,0.0,1.0
IDX94_4,17608.0,0.000170,0.013052,0.0,0.0,0.0,0.0,1.0
S410B_1,17608.0,0.000114,0.010657,0.0,0.0,0.0,0.0,1.0
S410B_2,17608.0,0.000170,0.013052,0.0,0.0,0.0,0.0,1.0
S410B_3,17608.0,0.000341,0.018457,0.0,0.0,0.0,0.0,1.0
S410B_4,17608.0,0.001079,0.032832,0.0,0.0,0.0,0.0,1.0
S410B_5,17608.0,0.003635,0.060181,0.0,0.0,0.0,0.0,1.0


In [57]:
continue_var = list(remove_items)
df_continue_var = aggregate_by_caseid_mean(Modulo1633_REC94_2024_M[continue_var + ['CASEID']], caseid_col="CASEID")
df_continue_var.head()


,CASEID,S422I,S411CA,S411BA,S411EA,S411DA
0,325503101 2,0.0,2.0,2.0,2.0,2.0
1,325504701 2,0.0,4.0,4.0,4.0,4.0
2,325505001 1,0.0,2.0,2.0,2.0,2.0
3,325508901 2,0.0,1.0,1.0,1.0,1.0
4,325509701 2,0.0,3.0,3.0,3.0,3.0


In [58]:
Modulo1633_REC94_2024_fil = df_dummy_variables_REC94.merge(df_no_dummy_list_REC94, how='left', on='CASEID')
Modulo1633_REC94_2024_fil= Modulo1633_REC94_2024_fil.merge(df_continue_var, how ='left', on ='CASEID')
Modulo1633_REC94_2024_fil.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17608 entries, 0 to 17607
Data columns (total 269 columns):
 #    Column         Non-Null Count  Dtype  
---   ------         --------------  -----  
 0    CASEID         17608 non-null  object 
 1    S413           17608 non-null  Int64  
 2    S426E          17608 non-null  Int64  
 3    S426GA         17608 non-null  Int64  
 4    S426GB         17608 non-null  Int64  
 5    S426GC         17608 non-null  Int64  
 6    S426GD         17608 non-null  Int64  
 7    S426GE         17608 non-null  Int64  
 8    S430D          17608 non-null  Int64  
 9    S427DA         17608 non-null  Int64  
 10   S427DB         17608 non-null  Int64  
 11   S427DC         17608 non-null  Int64  
 12   S427DD         17608 non-null  Int64  
 13   S427DE         17608 non-null  Int64  
 14   S427DF         17608 non-null  Int64  
 15   S427DG         17608 non-null  Int64  
 16   S427F          17608 non-null  Int64  
 17   S436C          17608 non-null

In [59]:
# File name only
output_file = "Modulo1633_REC94_2024_fil.csv"

# Go one level up from the current working directory
base_dir = os.path.dirname(os.getcwd())   # gives "c:\\Users\\linoc\\OneDrive\\Encoder\\03_partos"
output_dir = os.path.join(base_dir, "03_bases_intermediarios")

#Full path
output_path = os.path.join(output_dir, output_file)

# Save DataFrame
Modulo1633_REC94_2024_fil.to_csv(output_path, index=False, encoding="utf-8-sig")

In [ ]:
# Modulo1633_REC41_2024 = add_prefix_except_caseid(Modulo1633_REC41_2024, 'REC41')
# Modulo1633_REC94_2024 = add_prefix_except_caseid(Modulo1633_REC94_2024, 'REC94')


# # 1. Merge com Modulo1633_REC41_2024 usando CASEID e MIDX
# df_merged = target[['CASEID','BIDX']].merge(
#     Modulo1633_REC41_2024.drop_duplicates(subset=['CASEID', 'REC41_MIDX']),
#     left_on=['CASEID', 'BIDX'],
#     right_on=['CASEID', 'REC41_MIDX'],
#     how='left',
# #    suffixes=('', '_rec41')
# )

# # 2. Merge com Modulo1633_REC94_2024 usando CASEID e IDX94
# df_merged = df_merged.merge(
#     Modulo1633_REC94_2024.drop_duplicates(subset=['CASEID', 'REC94_IDX94']),
#     left_on=['CASEID', 'BIDX'],
#     right_on=['CASEID', 'REC94_IDX94'],
#     how='left',
# #    suffixes=('', '_rec94')
# )

# df_merged.info(verbose = True, show_counts = True)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21118 entries, 0 to 21117
Data columns (total 208 columns):
 #    Column          Non-Null Count  Dtype  
---   ------          --------------  -----  
 0    CASEID          21118 non-null  object 
 1    BIDX            21118 non-null  int64  
 2    REC41_ID1       19751 non-null  float64
 3    REC41_MIDX      19751 non-null  float64
 4    REC41_M1        19751 non-null  object 
 5    REC41_M1A       19751 non-null  object 
 6    REC41_M1B       19751 non-null  object 
 7    REC41_M1C       19751 non-null  object 
 8    REC41_M1D       19751 non-null  object 
 9    REC41_M1E       19751 non-null  object 
 10   REC41_M2A       19751 non-null  object 
 11   REC41_M2B       19751 non-null  object 
 12   REC41_M2C       19751 non-null  object 
 13   REC41_M2D       19751 non-null  object 
 14   REC41_M2E       19751 non-null  object 
 15   REC41_M2F       19751 non-null  object 
 16   REC41_M2G       19751 non-null  object 
 17   REC41_M2H 